# Feature Engineering - кейс № 6

Цель работы - подготовить признаки для прогноза средней активной мощности металлургического завода на следующие 15 минут.

В ноутбуке создаются 10 признаков: 4 лаговых, 2 скользящих средних и 4 календарных.

## 1. Загрузка и первичный осмотр данных

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [3]:
DATA_URL = 'https://raw.githubusercontent.com/red77789/team6/main/Steel_industry_data.csv'

df = pd.read_csv(DATA_URL)
df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


In [7]:
print(f'Размер исходной таблицы: {df.shape}')
display(df.info())
display(df.isna().sum())

Размер исходной таблицы: (35040, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35040 entries, 0 to 35039
Data columns (total 11 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   date                                  35040 non-null  object 
 1   Usage_kWh                             35040 non-null  float64
 2   Lagging_Current_Reactive.Power_kVarh  35040 non-null  float64
 3   Leading_Current_Reactive_Power_kVarh  35040 non-null  float64
 4   CO2(tCO2)                             35040 non-null  float64
 5   Lagging_Current_Power_Factor          35040 non-null  float64
 6   Leading_Current_Power_Factor          35040 non-null  float64
 7   NSM                                   35040 non-null  int64  
 8   WeekStatus                            35040 non-null  object 
 9   Day_of_week                           35040 non-null  object 
 10  Load_Type                             35040 n

None

,0
date,0
Usage_kWh,0
Lagging_Current_Reactive.Power_kVarh,0
Leading_Current_Reactive_Power_kVarh,0
CO2(tCO2),0
Lagging_Current_Power_Factor,0
Leading_Current_Power_Factor,0
NSM,0
WeekStatus,0
Day_of_week,0


## 2. Подготовка временного столбца

Сортировка по времени обязательна: лаги и скользящие средние должны использовать только прошлые наблюдения.

In [9]:
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y %H:%M')
df = df.sort_values('date').reset_index(drop=True)

intervals = df['date'].diff().dropna()
print('Начало данных:', df['date'].min())
print('Конец данных:', df['date'].max())
print('Основной интервал измерений:', intervals.mode().iloc[0])
print('Количество пропусков по времени:', (intervals != pd.Timedelta(minutes=15)).sum())

Начало данных: 2018-01-01 00:00:00
Конец данных: 2018-12-31 23:45:00
Основной интервал измерений: 0 days 00:15:00
Количество пропусков по времени: 0


## 3. Лаговые признаки

lag_1 - потребление 15 минут назад, а lag_96 - в то же время сутки назад.

In [10]:
df['lag_1'] = df['Usage_kWh'].shift(1)
df['lag_4'] = df['Usage_kWh'].shift(4)
df['lag_96'] = df['Usage_kWh'].shift(96)
df['lag_672'] = df['Usage_kWh'].shift(672)

## 4. Скользящие средние

Средние рассчитываются только по предыдущим значениям. Поэтому модель не получает информацию о потреблении в момент, который должна предсказать.

In [11]:
past_usage = df['Usage_kWh'].shift(1)
df['rolling_mean_4'] = past_usage.rolling(window=4).mean()
df['rolling_mean_24'] = past_usage.rolling(window=24).mean()

## 5. Календарные признаки

Время заранее известно, поэтому час, день недели, выходной и месяц можно использовать для прогноза.

In [12]:
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['month'] = df['date'].dt.month

## 6. Целевая переменная и итоговая таблица

Исходный показатель Usage_kWh измерен за 15 минут. Средняя активная мощность в кВт равна потреблённой энергии, умноженной на 4.

Текущее значение Usage_kWh не включаем в признаки: иначе модель увидела бы ответ заранее.

In [13]:
df['target_power_kw'] = df['Usage_kWh'] * 4

feature_columns = [
    'lag_1', 'lag_4', 'lag_96', 'lag_672',
    'rolling_mean_4', 'rolling_mean_24',
    'hour', 'day_of_week', 'is_weekend', 'month'
]

df_final = df[['date', *feature_columns, 'target_power_kw']].dropna().copy()

print(f'Размер итоговой таблицы: {df_final.shape}')
print(f'Пропусков в итоговой таблице: {df_final.isna().sum().sum()}')
df_final.head()

Размер итоговой таблицы: (34368, 12)
Пропусков в итоговой таблице: 0


,date,lag_1,lag_4,lag_96,lag_672,rolling_mean_4,rolling_mean_24,hour,day_of_week,is_weekend,month,target_power_kw
672,2018-01-08 00:00:00,3.42,3.64,3.28,3.42,3.4850,3.680833,0,0,0,1,18.72
673,2018-01-08 00:15:00,4.68,3.24,3.13,3.17,3.7450,3.730417,0,0,0,1,15.12
674,2018-01-08 00:30:00,3.78,3.64,3.49,4.00,3.8800,3.748333,0,0,0,1,13.52
675,2018-01-08 00:45:00,3.38,3.42,3.64,3.24,3.8150,3.724167,0,0,0,1,13.24
676,2018-01-08 01:00:00,3.31,4.68,3.20,3.31,3.7875,3.713750,1,0,0,1,15.56


## 7. Результат

Получена таблица из 34 368 наблюдений: дата, 10 признаков и целевая переменная target_power_kw.

Для корректной проверки модели данные нужно разделять по времени: ранние даты использовать для обучения, поздние - для проверки. Полиномиальные признаки и регуляризацию следует проверять в отдельном ноутбуке с моделью.

In [14]:
OUTPUT_FILE = 'steel_features_power_team6.csv'
df_final.to_csv(OUTPUT_FILE, index=False)
print(f'Файл сохранён: {OUTPUT_FILE}')


Файл сохранён: steel_features_power_team6.csv
